# Mental Health & Burnout in Tech (2026)
## End-to-End ML Pipeline — with Hyperparameter Tuning & Cross-Validation

**Dataset:** `mental_health_burnout_tech_2026.csv` — 100,000 records, 36 columns, no missing values.

**Goal:** Predict `burnout_level` (Low / Moderate / High / Severe) using an automated pipeline that:
- Accepts the full dataset; you only specify `FEATURE_COLS` and `TARGET`
- Handles train/test splitting internally
- Tunes hyperparameters automatically with `RandomizedSearchCV`
- Validates stability with `StratifiedKFold` cross-validation

### Pipeline steps
| # | Step |
| - | ---- |
| 1 | Load & inspect |
| 2 | Missing-value check + drop duplicates |
| 3 | Univariate analysis & visualisations |
| 4 | Outlier detection (IQR multi-column rule) |
| 5 | Feature engineering (19 derived features) |
| 6 | Build feature matrix from `FEATURE_COLS` + `TARGET` |
| 7 | `RandomizedSearchCV` on HGB & XGBoost (10 iter × 5-fold CV) |
| 8 | Soft-voting ensemble + holdout evaluation |
| 9 | Save artefacts for the Streamlit dashboard |

## 1 · Imports & Display Settings

In [ ]:
%matplotlib inline
import json, time, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                      cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (LabelEncoder, OneHotEncoder,
                                    StandardScaler, TargetEncoder)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from feature_engineering import MODEL_CATEGORICAL_COLS, RAW_CATEGORICAL_COLS, engineer_features
from models import SoftVotingEnsemble

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 4)})
print('All imports OK')

## 2 · User Configuration

**This is the only section you need to edit.** Set the feature columns,
target, and tuning budget, then run all cells.

In [ ]:
# ── USER SETTINGS ───────────────────────────────────────────────────────
#
# FEATURE_COLS : list of column names to use as model inputs.
#                Set to None to auto-select all non-leakage, non-target columns.
FEATURE_COLS = None   # e.g. ['stress_score', 'work_hours_per_week', ...]

# TARGET : the column the model predicts.
TARGET = 'burnout_level'

# Tuning budget — more iterations = better params but longer runtime.
# 10 iterations x 5 folds ~ 5-10 min on a modern laptop.
N_ITER_SEARCH = 10
CV_FOLDS      = 5

# ── Internal constants (no need to change) ───────────────────────────────
RANDOM_STATE = 42
HERE         = Path().resolve()
CSV_PATH     = HERE / 'mental_health_burnout_tech_2026.csv'
ART_DIR      = HERE / 'artifacts'
ART_DIR.mkdir(exist_ok=True)

LEAKAGE_COLS   = ['burnout_score','phq9_score','phq9_category',
                  'gad7_score','gad7_category',
                  'seeks_mental_health_support','job_change_intention']
ID_COLS        = ['employee_id']
LOW_CARD_CATS  = ['gender', 'work_mode']
HIGH_CARD_CATS = ['country', 'job_role', 'industry']
IQR_COLS       = ['salary_usd','work_hours_per_week','meetings_per_day','team_size',
                  'sleep_hours_per_night','vacation_days_taken',
                  'years_experience','years_at_company']

print('Config ready. Target:', TARGET)
print('Feature selection:', 'auto (all non-leakage)' if FEATURE_COLS is None else FEATURE_COLS)

## 3 · Load & Inspect the Data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Shape : {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
print('--- dtypes ---')
print(df.dtypes.value_counts())
print()
df.describe().T[['mean','std','min','max']].round(2)

## 4 · Data Quality: Missing Values & Duplicates

`SimpleImputer` steps remain inside the `ColumnTransformer` as a safety net
for any future data that might contain gaps.

In [ ]:
miss = df.isnull().sum()
print('Columns with missing values:')
print(miss[miss > 0] if miss.any() else '  None found')

before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c not in ID_COLS]).reset_index(drop=True)
print(f'\nRows before / after dedup: {before:,} -> {len(df):,}  ({before-len(df)} removed)')

## 5 · Univariate Analysis

### 5.1 Target distribution

In [ ]:
CLASS_ORDER  = ['Low','Moderate','High','Severe']
CLASS_COLORS = {'Low':'#27ae60','Moderate':'#f39c12','High':'#e67e22','Severe':'#e74c3c'}

vc  = df[TARGET].value_counts().reindex(CLASS_ORDER)
pct = vc / vc.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(vc.index, vc.values,
              color=[CLASS_COLORS[l] for l in vc.index], edgecolor='white', linewidth=1.2)
for bar, p in zip(bars, pct):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+180,
            f'{p:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set(title='Burnout Level Distribution', xlabel='Level', ylabel='Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout(); plt.show()
print(vc.to_string())

### 5.2 Numeric feature distributions

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop(ID_COLS).tolist()
n_cols, n_rows = 4, -(-len(num_cols)//4)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', alpha=0.75, edgecolor='white')
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Numeric Feature Distributions', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 5.3 Correlation heatmap

In [ ]:
key_cols = ['age','years_experience','work_hours_per_week','sleep_hours_per_night',
            'exercise_days_per_week','vacation_days_taken','manager_support_score',
            'work_life_balance_score','job_satisfaction_score','deadline_pressure_score',
            'autonomy_score','stress_score']
corr = df[key_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size':8})
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=12)
plt.tight_layout(); plt.show()

## 6 · Outlier Detection (IQR Multi-Column Rule)

A row is dropped only if it is an IQR outlier in **>=2** of the bounded numeric columns.

In [ ]:
bounds   = {}
flag_mat = np.zeros((len(df), len(IQR_COLS)), dtype=bool)
for j, col in enumerate(IQR_COLS):
    q1, q3  = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr_val = q3 - q1
    lo, hi  = q1 - 1.5*iqr_val, q3 + 1.5*iqr_val
    bounds[col] = {'low':float(lo),'high':float(hi),'q1':float(q1),'q3':float(q3)}
    flag_mat[:, j] = (df[col] < lo) | (df[col] > hi)
drop_mask = flag_mat.sum(axis=1) >= 2
df = df.loc[~drop_mask].reset_index(drop=True)
print(f'Dropped {int(drop_mask.sum()):,} rows ({drop_mask.mean():.2%}) -> {len(df):,} remaining')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for ax, col in zip(axes, IQR_COLS):
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=9); ax.tick_params(labelsize=8)
plt.suptitle('IQR Columns After Outlier Removal', fontsize=12)
plt.tight_layout(); plt.show()

## 7 · Feature Engineering

All 19 derived features are row-wise transforms — no statistics learned from data —
so they can be applied before the train/test split without leakage.

| Group | Features |
| ----- | -------- |
| Ordinal encoding | `seniority_ord`, `company_size_ord` |
| Composite interactions | `work_recovery_ratio`, `pressure_vs_support`, `resilience_index`, `tenure_experience_ratio`, `salary_per_experience`, `meetings_burden` |
| Log transforms | `log_salary`, `log_team_size` |
| Binary flags | `overworked`, `sleep_deprived`, `no_vacation`, `high_meetings`, `no_exercise` |
| Polynomial cross-products | `stress_x_hours`, `stress_x_sleep_inv`, `pressure_x_imbalance`, `stress_x_no_support` |

In [ ]:
raw_df = df.copy()
eng_df = engineer_features(df)
derived = [c for c in eng_df.columns if c not in df.columns]
print(f'Added {len(derived)} engineered features:')
for f in derived: print(f'  {f}')

In [ ]:
# Visualise two key engineered features by burnout level
plot_df = eng_df.copy()
plot_df['burnout_level'] = df[TARGET]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ['work_recovery_ratio', 'pressure_vs_support']):
    for level in CLASS_ORDER:
        vals = plot_df.loc[plot_df['burnout_level']==level, col]
        ax.hist(vals, bins=50, alpha=0.5, label=level, density=True,
                color=CLASS_COLORS[level])
    ax.set_title(col, fontsize=11); ax.legend(fontsize=8)
plt.suptitle('Key Engineered Features by Burnout Level', fontsize=12)
plt.tight_layout(); plt.show()
del plot_df

## 8 · Build Feature Matrix from FEATURE_COLS + TARGET

The pipeline auto-detects which columns to use based on your `FEATURE_COLS` setting.
The 80/20 stratified split is handled internally — you never need to do it manually.

In [ ]:
# Resolve feature columns
if FEATURE_COLS is None:
    auto_raw = [c for c in raw_df.columns if c not in LEAKAGE_COLS+ID_COLS+[TARGET]]
    print(f'FEATURE_COLS=None -> auto-selected {len(auto_raw)} raw columns')
else:
    missing = [c for c in FEATURE_COLS if c not in raw_df.columns]
    if missing: raise ValueError(f'Columns not in dataset: {missing}')
    auto_raw = FEATURE_COLS
    print(f'Using {len(auto_raw)} user-specified columns')

# Build feature_df: all engineered numerics + model categoricals
drop_all   = LEAKAGE_COLS + ID_COLS + [TARGET] + RAW_CATEGORICAL_COLS
feature_df = eng_df.drop(columns=drop_all)
raw_cats_in_use   = [c for c in auto_raw if c in RAW_CATEGORICAL_COLS]
raw_nums_in_use   = [c for c in auto_raw if c not in RAW_CATEGORICAL_COLS]
eng_numeric       = [c for c in derived if c in feature_df.columns]
raw_num_present   = [c for c in raw_nums_in_use if c in feature_df.columns]
feature_df        = feature_df[raw_num_present + eng_numeric]
model_cats_in_use = [c for c in MODEL_CATEGORICAL_COLS if c in raw_cats_in_use] or MODEL_CATEGORICAL_COLS
feature_df        = pd.concat([feature_df, eng_df[model_cats_in_use]], axis=1)

model_numeric_cols   = [c for c in feature_df.columns if c not in model_cats_in_use]
raw_numeric_cols_out = [c for c in model_numeric_cols if c not in derived]

print(f'Feature matrix : {feature_df.shape}')
print(f'  Numeric cols  : {len(model_numeric_cols)}')
print(f'  Categoric cols: {len(model_cats_in_use)}')

# Label-encode the target
le = LabelEncoder()
y  = le.fit_transform(df[TARGET])
print(f'\nClass mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Stratified 80/20 split (internal — user just provides the full dataset)
X_train, X_test, y_train, y_test = train_test_split(
    feature_df, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}  (80/20 stratified)')

## 9 · Preprocessing Pipeline

In [ ]:
def build_preprocessor(numeric_cols, categorical_cols):
    """Three-branch ColumnTransformer: numeric / OHE (low-card) / TargetEncoder (high-card)."""
    numeric_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ])
    ohe_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    target_enc_pipe = Pipeline([
        ('impute',     SimpleImputer(strategy='most_frequent')),
        ('target_enc', TargetEncoder(target_type='multiclass',
                                     smooth='auto', random_state=RANDOM_STATE)),
    ])
    low_card  = [c for c in categorical_cols if c in LOW_CARD_CATS]
    high_card = [c for c in categorical_cols if c in HIGH_CARD_CATS]
    transformers = [('num', numeric_pipe, numeric_cols)]
    if low_card:  transformers.append(('ohe',    ohe_pipe,        low_card))
    if high_card: transformers.append(('target', target_enc_pipe, high_card))
    return ColumnTransformer(transformers)

print('build_preprocessor() defined')

## 10 · Hyperparameter Tuning with RandomizedSearchCV

Each model is tuned automatically:
- **`RandomizedSearchCV`** samples `N_ITER_SEARCH` random hyperparameter combinations
- **`StratifiedKFold`** splits keep the class balance intact in every fold
- Class-balanced weighting corrects for the smaller `High` class

| Model | Params tuned |
| ----- | ------------ |
| HistGradientBoosting | learning rate, depth, leaf size, L2 reg, iterations |
| XGBoost | learning rate, depth, subsample, column sample, L1 reg, estimators |
| LogisticRegression | baseline only — no tuning needed |

In [ ]:
cv             = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
sample_weights = compute_sample_weight('balanced', y_train)

# ── Tune HistGradientBoosting ────────────────────────────────────────────
print(f'[1/3] Tuning HistGradientBoosting ({N_ITER_SEARCH} iter x {CV_FOLDS}-fold) ...')
t0 = time.time()
hgb_search = RandomizedSearchCV(
    Pipeline([
        ('pre', build_preprocessor(model_numeric_cols, model_cats_in_use)),
        ('clf', HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    param_distributions={
        'clf__max_iter':          randint(200, 600),
        'clf__learning_rate':     loguniform(0.02, 0.20),
        'clf__max_depth':         [4, 5, 6, 7, 8],
        'clf__min_samples_leaf':  randint(10, 60),
        'clf__l2_regularization': loguniform(0.001, 2.0),
    },
    n_iter=N_ITER_SEARCH, cv=cv, scoring='f1_macro',
    n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=0,
)
hgb_search.fit(X_train, y_train)
print(f'  Best CV macro-F1: {hgb_search.best_score_:.3f}  ({time.time()-t0:.0f}s)')
print(f'  Best params: {hgb_search.best_params_}')

In [ ]:
# ── Tune XGBoost ─────────────────────────────────────────────────────────
print(f'[2/3] Tuning XGBoost ({N_ITER_SEARCH} iter x {CV_FOLDS}-fold) ...')
t0 = time.time()
xgb_search = RandomizedSearchCV(
    Pipeline([
        ('pre', build_preprocessor(model_numeric_cols, model_cats_in_use)),
        ('clf', XGBClassifier(eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
    param_distributions={
        'clf__n_estimators':     randint(200, 500),
        'clf__learning_rate':    loguniform(0.02, 0.15),
        'clf__max_depth':        randint(4, 9),
        'clf__subsample':        [0.6, 0.7, 0.8, 0.9],
        'clf__colsample_bytree': [0.6, 0.7, 0.8, 0.9],
        'clf__reg_alpha':        loguniform(0.001, 1.0),
    },
    n_iter=N_ITER_SEARCH, cv=cv, scoring='f1_macro',
    n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=0,
)
xgb_search.fit(X_train, y_train, clf__sample_weight=sample_weights)
print(f'  Best CV macro-F1: {xgb_search.best_score_:.3f}  ({time.time()-t0:.0f}s)')
print(f'  Best params: {xgb_search.best_params_}')

In [ ]:
# ── Baseline LogisticRegression ──────────────────────────────────────────
print('[3/3] Fitting baseline LogisticRegression ...')
t0 = time.time()
lr_pipe = Pipeline([
    ('pre', build_preprocessor(model_numeric_cols, model_cats_in_use)),
    ('clf', LogisticRegression(max_iter=2000, n_jobs=-1, C=3.0, class_weight='balanced')),
])
lr_pipe.fit(X_train, y_train)
print(f'  Done ({time.time()-t0:.1f}s)')

## 11 · Holdout Evaluation & Leaderboard

In [ ]:
fitted_pipes = {
    'LogisticRegression':  lr_pipe,
    'HistGradientBoosting': hgb_search.best_estimator_,
    'XGBoost':             xgb_search.best_estimator_,
}
leaderboard = {}
best_name, best_obj, best_f1 = None, None, -1.0

for name, pipe in fitted_pipes.items():
    preds = pipe.predict(X_test)
    f1    = f1_score(y_test, preds, average='macro')
    acc   = accuracy_score(y_test, preds)
    leaderboard[name] = {'macro_f1': float(f1), 'accuracy': float(acc)}
    print(f'  {name:<26s}  acc={acc:.3f}  macro-F1={f1:.3f}')
    if f1 > best_f1: best_f1, best_name, best_obj = f1, name, pipe

# Soft-voting ensemble
ensemble  = SoftVotingEnsemble(list(fitted_pipes.values()))
ens_preds = ensemble.predict(X_test)
ens_f1    = f1_score(y_test, ens_preds, average='macro')
ens_acc   = accuracy_score(y_test, ens_preds)
leaderboard['Ensemble(LR+HGB+XGB)'] = {'macro_f1': float(ens_f1), 'accuracy': float(ens_acc)}
print(f"  {'Ensemble(LR+HGB+XGB)':<26s}  acc={ens_acc:.3f}  macro-F1={ens_f1:.3f}")
if ens_f1 > best_f1: best_f1, best_name, best_obj = ens_f1, 'Ensemble(LR+HGB+XGB)', ensemble
print(f'\nWinner -> {best_name}  (macro-F1 = {best_f1:.3f})')

In [ ]:
# Leaderboard bar chart
names = list(leaderboard.keys())
f1s   = [leaderboard[n]['macro_f1'] for n in names]
accs  = [leaderboard[n]['accuracy']  for n in names]
x, w  = np.arange(len(names)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, accs, w, label='Accuracy',  color='steelblue',  alpha=0.85)
b2 = ax.bar(x+w/2, f1s,  w, label='Macro-F1', color='darkorange', alpha=0.85)
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=12, ha='right')
ax.set_ylim(0.55, 0.80)
ax.set(title='Model Comparison - Accuracy vs Macro-F1', ylabel='Score')
ax.legend(); plt.tight_layout(); plt.show()

## 12 · Cross-Validation Stability Check

Run the best individual model through 5 independent test folds.
Bars close together = the model is reliable, not a one-off lucky result.

In [ ]:
best_ind_name = max([(n,p) for n,p in fitted_pipes.items()],
                    key=lambda x: leaderboard[x[0]]['macro_f1'])[0]
cv_fold_scores = cross_val_score(
    fitted_pipes[best_ind_name], X_train, y_train,
    cv=cv, scoring='f1_macro', n_jobs=-1
)
print(f'CV model    : {best_ind_name}')
print(f'Fold scores : {[round(s,3) for s in cv_fold_scores]}')
print(f'Mean +/- Std: {cv_fold_scores.mean():.3f} +/- {cv_fold_scores.std():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cv_mean = cv_fold_scores.mean()
colors  = ['#27ae60' if s >= cv_mean else '#e74c3c' for s in cv_fold_scores]
bars    = ax.bar([f'Fold {i+1}' for i in range(len(cv_fold_scores))],
                 cv_fold_scores * 100, color=colors, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=2, fontsize=10)
ax.axhline(cv_mean*100, ls='--', color='#2c3e50', lw=2,
           label=f'Average: {cv_mean*100:.1f}%')
ax.set(title=f'Cross-Validation Fold Scores ({best_ind_name})',
       ylabel='Macro-F1 (%)',
       ylim=[max(0, cv_fold_scores.min()*100-5), min(100, cv_fold_scores.max()*100+5)])
ax.legend(); plt.tight_layout(); plt.show()

## 13 · Evaluate the Winning Model

### 13.1 Classification report

In [ ]:
# Auto-recovery: rebuild any missing variables so this section runs
# even if earlier cells were skipped or the kernel was restarted.
import joblib as _jl

_need_model = any(v not in vars() for v in ["best_obj", "le", "best_name"])
_need_data  = any(v not in vars() for v in ["X_test", "y_test"])

if _need_model:
    print("Loading model artefacts from disk ...")
    best_obj  = _jl.load(ART_DIR / "model.joblib")
    le        = _jl.load(ART_DIR / "label_encoder.joblib")
    _m        = json.loads((ART_DIR / "metrics.json").read_text())
    best_name = _m["winning_model"]
    print(f"  Model  : {best_name}")

if _need_data:
    print("Rebuilding test set (same seed=42 split) ...")
    _feat_meta     = json.loads((ART_DIR / "feature_meta.json").read_text())
    _feature_order = _feat_meta["feature_order"]
    # Reload and clean data
    _df = pd.read_csv(CSV_PATH)
    _df = _df.drop_duplicates(
        subset=[c for c in _df.columns if c not in ID_COLS]
    ).reset_index(drop=True)
    # IQR outlier removal (same rule as Step 6)
    _flag = np.zeros((len(_df), len(IQR_COLS)), dtype=bool)
    for _j, _col in enumerate(IQR_COLS):
        _q1, _q3 = _df[_col].quantile(0.25), _df[_col].quantile(0.75)
        _iqr = _q3 - _q1
        _flag[:, _j] = (_df[_col] < _q1-1.5*_iqr) | (_df[_col] > _q3+1.5*_iqr)
    _df = _df.loc[_flag.sum(axis=1) < 2].reset_index(drop=True)
    # Feature engineering
    _eng = engineer_features(_df)
    # Build feature matrix in the exact column order the model expects
    _drop    = LEAKAGE_COLS + ID_COLS + [TARGET] + RAW_CATEGORICAL_COLS
    _feat_df = _eng.drop(columns=[c for c in _drop if c in _eng.columns])
    _cats    = [c for c in MODEL_CATEGORICAL_COLS if c in _eng.columns]
    _feat_df = pd.concat([_feat_df, _eng[_cats]], axis=1)
    _feat_df = _feat_df[[c for c in _feature_order if c in _feat_df.columns]]
    _y       = le.transform(_df[TARGET])
    _, X_test, _, y_test = train_test_split(
        _feat_df, _y, test_size=0.20, stratify=_y, random_state=RANDOM_STATE
    )
    print(f"  X_test : {X_test.shape}")

labels = le.classes_.tolist()
print(f"Ready  -> model={type(best_obj).__name__}  X_test={X_test.shape}")


In [ ]:
preds       = best_obj.predict(X_test)
acc         = accuracy_score(y_test, preds)
macro_f1    = f1_score(y_test, preds, average='macro')
weighted_f1 = f1_score(y_test, preds, average='weighted')
report_dict = classification_report(
    y_test, preds, target_names=le.classes_.tolist(), output_dict=True, zero_division=0
)
print(f'Winner       : {best_name}')
print(f'Accuracy     : {acc:.4f}')
print(f'Macro F1     : {macro_f1:.4f}')
print(f'Weighted F1  : {weighted_f1:.4f}')
print(f'CV mean F1   : {cv_fold_scores.mean():.4f} +/- {cv_fold_scores.std():.4f}')
print()
print(classification_report(y_test, preds, target_names=le.classes_.tolist(), zero_division=0))

### 13.2 Confusion matrix

In [ ]:
cm     = confusion_matrix(y_test, preds)
labels = le.classes_.tolist()
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_pct, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set(title=f'Confusion Matrix (%) - {best_name}',
       xlabel='Predicted', ylabel='True')
plt.tight_layout(); plt.show()

### 13.3 Per-class F1 with colour coding

In [ ]:
class_f1s = [report_dict[c]['f1-score'] for c in labels]
bar_colors = ['#27ae60' if f>=0.75 else '#f39c12' if f>=0.55 else '#e74c3c'
              for f in class_f1s]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f'{c}' for c in labels], class_f1s,
              color=[CLASS_COLORS[c] for c in labels], edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=11, fontweight='bold')
ax.axhline(macro_f1, ls='--', color='navy', lw=1.8,
           label=f'Macro avg = {macro_f1:.3f}')
ax.axhline(0.75, ls=':', color='#27ae60', lw=1.4, label='Great threshold (0.75)')
ax.set(title='Per-Class F1 Score', xlabel='Burnout Level',
       ylabel='F1 Score', ylim=(0, 1.05))
ax.legend(); plt.tight_layout(); plt.show()

### 13.4 Permutation importance (top 15 features)

In [ ]:
perm_model  = best_obj.pipelines[0] if isinstance(best_obj, SoftVotingEnsemble) else best_obj
sample_size = min(3000, len(X_test))
idx         = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=sample_size, replace=False)
perm = permutation_importance(
    perm_model, X_test.iloc[idx], y_test[idx],
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=1, scoring='f1_macro'
)
imp_df = pd.DataFrame({
    'feature':    X_test.columns,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=False).head(15)

ENGINEERED = {'stress_x_hours','stress_x_sleep_inv','pressure_x_imbalance',
              'stress_x_no_support','work_recovery_ratio','resilience_index',
              'pressure_vs_support','seniority_ord','company_size_ord',
              'log_salary','log_team_size','overworked','sleep_deprived',
              'no_vacation','high_meetings','no_exercise','meetings_burden',
              'tenure_experience_ratio','salary_per_experience'}
colors = ['tomato' if f in ENGINEERED else 'steelblue' for f in imp_df['feature']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
        color=colors[::-1], xerr=imp_df['std'][::-1], capsize=3, alpha=0.85)
ax.set(title='Top 15 Features - Permutation Importance',
       xlabel='Mean decrease in macro-F1')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='tomato',    label='Engineered'),
                   Patch(color='steelblue', label='Original')], loc='lower right')
plt.tight_layout(); plt.show()
print(imp_df.to_string(index=False))

## 14 · Save Artefacts

In [ ]:
def _clean(v):
    return float(v) if hasattr(v, 'item') else v

best_params = {
    'HistGradientBoosting': {k.replace('clf__',''):_clean(v) for k,v in hgb_search.best_params_.items()},
    'XGBoost':              {k.replace('clf__',''):_clean(v) for k,v in xgb_search.best_params_.items()},
}
importance_list = [
    {'feature': r['feature'], 'importance': float(r['importance']), 'std': float(r['std'])}
    for r in imp_df.to_dict('records')
]

metrics = {
    'winning_model':          best_name,
    'accuracy':               float(acc),
    'macro_f1':               float(macro_f1),
    'weighted_f1':            float(weighted_f1),
    'cv_fold_scores':         [float(s) for s in cv_fold_scores],
    'cv_mean':                float(cv_fold_scores.mean()),
    'cv_std':                 float(cv_fold_scores.std()),
    'cv_model':               best_ind_name,
    'best_params':            best_params,
    'classification_report':  report_dict,
    'confusion_matrix':       cm.tolist(),
    'class_labels':           labels,
    'permutation_importance': importance_list,
    'leaderboard':            leaderboard,
    'search_scores': {
        'HistGradientBoosting': {
            'mean_cv_scores': [float(v) for v in hgb_search.cv_results_['mean_test_score']],
            'std_cv_scores':  [float(v) for v in hgb_search.cv_results_['std_test_score']],
        },
        'XGBoost': {
            'mean_cv_scores': [float(v) for v in xgb_search.cv_results_['mean_test_score']],
            'std_cv_scores':  [float(v) for v in xgb_search.cv_results_['std_test_score']],
        },
    },
}

feature_meta = {'numeric':{},'categorical':{},'derived_numeric':{},
                'iqr_bounds': bounds,
                'feature_order': model_numeric_cols + model_cats_in_use}
for col in raw_numeric_cols_out:
    s = raw_df[col]
    feature_meta['numeric'][col] = {
        'min':float(s.min()),'max':float(s.max()),'median':float(s.median()),
        'is_integer':bool(pd.api.types.is_integer_dtype(s))}
for col in RAW_CATEGORICAL_COLS:
    feature_meta['categorical'][col] = sorted(raw_df[col].dropna().unique().tolist())
for col in [c for c in model_numeric_cols if c not in raw_numeric_cols_out]:
    s = eng_df[col]
    feature_meta['derived_numeric'][col] = {
        'min':float(s.min()),'max':float(s.max()),'median':float(s.median())}

joblib.dump(best_obj, ART_DIR / 'model.joblib')
joblib.dump(le,       ART_DIR / 'label_encoder.joblib')
(ART_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
(ART_DIR / 'feature_meta.json').write_text(json.dumps(feature_meta, indent=2))

print('Saved artefacts:')
for p in sorted(ART_DIR.iterdir()):
    print(f'  {p.name:<30s} {p.stat().st_size/1024:>6.0f} KB')

---
## Done

Artefacts in `artifacts/` are ready for the Streamlit dashboard:
```
artifacts/
  model.joblib           <- best model / ensemble
  label_encoder.joblib   <- maps 0-3 -> Low/Moderate/High/Severe
  metrics.json           <- accuracy, F1, CV scores, best params, importance
  feature_meta.json      <- slider ranges + dropdown options for predict form
  univariate.json        <- pre-computed stats for EDA page
```

Launch the dashboard:
```bash
streamlit run app.py
```